In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

# 1. Load Dataset
data = pd.read_csv("C:/Users/cokre/Downloads/Compressed/weatherHistory.csv")

# Fix datetime warning
data['Formatted Date'] = pd.to_datetime(data['Formatted Date'], utc=True)

# Sort by date
data = data.sort_values('Formatted Date')

# 2. Select Temperature Column
dataset = data[['Temperature (C)']].values

# 3. Normalize
scaler = MinMaxScaler()
dataset_scaled = scaler.fit_transform(dataset)

# 4. Create Sequences
def create_dataset(data, time_step=10):
    X, y = [], []
    for i in range(len(data) - time_step):
        X.append(data[i:i+time_step])
        y.append(data[i+time_step])
    return np.array(X), np.array(y)

time_step = 10
X, y = create_dataset(dataset_scaled, time_step)

# 5. Train-Test Split
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 6. Build Model (fixed warning using Input layer)
model = Sequential([
    Input(shape=(time_step, 1)),
    LSTM(50),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')

# 7. Train
model.fit(X_train, y_train, epochs=10, batch_size=32)

# 8. Predict
predictions = model.predict(X_test)

# 9. Inverse Scaling
predictions = scaler.inverse_transform(predictions)
y_test_actual = scaler.inverse_transform(y_test)

# 10. Plot
plt.plot(y_test_actual, label="Actual Temp")
plt.plot(predictions, label="Predicted Temp")
plt.legend()
plt.title("Temperature Forecast using LSTM")
plt.show()